# Korea Pine-Wilt Fisher-KPP Simulation

??? ??????? ?? ?? compact CSV/NPZ? ??, 2016? ?? ???? ?????? 2D Fisher-KPP RK4 ?? ?????? ?????. ?? ??? CSV? GitHub ?? ?? ??? ?? ??? ? ???? repo? ??? compact ??? ??? ?? checksum manifest? ?????.

## 1. Setup

Colab??? repo? ?? ?? ????? ??? GitHub?? ?? clone? ?????. ????? ? ???? repository root ?? ? ?? ???? ???? ???.

In [ ]:
%matplotlib inline

from __future__ import annotations

import csv
import json
import subprocess
import sys
from argparse import Namespace
from pathlib import Path

from IPython.display import Image, Markdown, display


def _has_project(root: Path) -> bool:
    return (
        (root / "fisher_origin_lab").exists()
        and (root / "data" / "korea_pine_wilt" / "processed" / "manifest.json").exists()
    )

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if _has_project(candidate):
        PROJECT_ROOT = candidate
        break

if not _has_project(PROJECT_ROOT) and Path("/content").exists():
    repo_dir = Path("/content/fisher-pinn")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/rladbsco24/fisher-pinn.git", str(repo_dir)],
            check=True,
        )
    PROJECT_ROOT = repo_dir.resolve()

if not _has_project(PROJECT_ROOT):
    raise RuntimeError(
        "Project files were not found. Run this notebook from the fisher-pinn repository root "
        "or use Colab with network access so the repository can be cloned."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"project root: {PROJECT_ROOT}")

## 2. Configuration

???? ????? ? ? ?? ??? ? ?? ?????. ? ??? ??? ???? `GRID_SIZE`? `STEPS_PER_YEAR`? ??? ???.

In [ ]:
from fisher_origin_lab.korea_data import load_manifest, load_korea_pine_wilt_points
from scripts.run_korea_pine_wilt_simulation import run as run_korea_pine_wilt_simulation

OUT_DIR = PROJECT_ROOT / "runs" / "korea_pine_wilt_notebook"
GRID_SIZE = 96
DIFFUSION = 0.0015
REACTION = 0.70
STEPS_PER_YEAR = 80
END_YEAR = 2030

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output dir: {OUT_DIR}")

## 3. Dataset Manifest

Compact ??? ?? ?? ??? ??? ????. ?? ??? CSV? ??? sha256? manifest? ?? ??? ???? ???? ????.

In [ ]:
manifest = load_manifest()
compact = manifest["compact_files"]
raw_by_year = {entry["year"]: entry for entry in manifest["raw_files"]}

rows = ["| year | compact records | raw CSV MB | raw sha256 prefix |", "|---:|---:|---:|---|"]
for year, count in compact["year_counts"].items():
    raw = raw_by_year[int(year)]
    rows.append(f"| {year} | {int(count):,} | {raw['bytes'] / 1_000_000:.1f} | `{raw['sha256'][:12]}` |")

display(Markdown("\n".join(rows)))
print(json.dumps(manifest["source"], indent=2, ensure_ascii=False))

## 4. Load Compact Observation Points

???? manifest ?? `EPSG:5179`???. ? ??? 318? ? ???? ???? ????.

In [ ]:
points = load_korea_pine_wilt_points()
print(f"records: {len(points.year):,}")
print(f"years: {int(points.year.min())}..{int(points.year.max())}")
print(f"x range: {points.x.min():.1f}..{points.x.max():.1f}")
print(f"y range: {points.y.min():.1f}..{points.y.max():.1f}")
print(f"crs: {points.crs}")

## 5. Run 2D Fisher-KPP RK4 Simulation

2016? ?? density grid? ?????? ????, no-flux ????? ???? 2D Fisher-KPP ???? RK4? ?????.

In [ ]:
summary = run_korea_pine_wilt_simulation(
    Namespace(
        grid_size=GRID_SIZE,
        pad_m=15_000.0,
        capacity_percentile=99.0,
        smooth_passes=1,
        diffusion=DIFFUSION,
        reaction=REACTION,
        steps_per_year=STEPS_PER_YEAR,
        end_year=END_YEAR,
        output_dir=OUT_DIR,
    )
)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 6. Observed-Year Metrics

?? ?? ??? ?? 2016-2023?? ?? ?????? ?? density grid? ??? ????. ? RK4? real data calibration? ??? ?? PDE ?? ??? ?? baseline???.

In [ ]:
metrics_path = OUT_DIR / "korea_pine_wilt_metrics.csv"
with metrics_path.open("r", encoding="utf-8", newline="") as f:
    metric_rows = list(csv.DictReader(f))

md_rows = ["| year | relative L2 | correlation | observed mean | simulated mean |", "|---:|---:|---:|---:|---:|"]
for row in metric_rows:
    md_rows.append(
        "| {year} | {l2:.4f} | {corr:.4f} | {obs:.4f} | {sim:.4f} |".format(
            year=row["year"],
            l2=float(row["relative_l2"]),
            corr=float(row["correlation"]),
            obs=float(row["observed_mean"]),
            sim=float(row["simulated_mean"]),
        )
    )

display(Markdown("\n".join(md_rows)))

## 7. Visual Outputs

????? ??? PNG? ??? ??? ?????. ??? `runs/korea_pine_wilt_notebook`? ?????.

In [ ]:
for image_name in [
    "observed_density_by_year.png",
    "rk4_forecast_timeline.png",
    "observed_vs_simulated_metrics.png",
]:
    image_path = OUT_DIR / image_name
    display(Markdown(f"### `{image_name}`"))
    display(Image(filename=str(image_path)))

## 8. Notes

? ???? ??? ?? ??? Fisher-KPP forward simulation?? ???? ?? ??? baseline???. ?? ??? calibration?? ????? ??? reporting intensity, ??/?? ??, ??/?? mask, ??? ?? ?? ??, ??? spatially varying diffusion/reaction? ?? ????? ???.